In [ ]:
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
from openmmtools.integrators import LangevinIntegrator, LangevinSplittingGirsanov
import matplotlib

import torch
from torch import nn, optim, autograd
from torch.nn import functional as F
from torch.utils.data.dataset import random_split
from openmmtorch import TorchForce

import numpy as np
from matplotlib import pyplot as plt

import gc

from scipy.stats import gaussian_kde
from scipy.interpolate import CubicSpline
from scipy.spatial import cKDTree

from pymbar import MBAR

from potential import *
from fmrc import *

import warnings
warnings.filterwarnings('ignore')

In [ ]:
##############################################################################
# Global parameters
##############################################################################
mass = 1.0 * dalton
temp = 298.15
temperature = temp * kelvin
collision_rate = 10 / picosecond
timestep = 5 * femtosecond
splitting = 'R V O V R'
nstxout = 20
kt = temp*8.314/1000
platform = Platform.getPlatformByName('CUDA')

#### 1. Metadynamics Rerun


##### 1.1 2d Muller potential

In [ ]:
#%%time
# Biasfactor
biasfactor = 2
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000
 
torchforce_savedir = 'models/muller_2d_torchforce.pt'
bias = np.load('bias/metad_rerun_bias.npy')

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))
    
    save_dir = 'traj_and_dat/2d_muller/metad_rerun/{n_iters}_gamma={biasfactor}_metad_rerun_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    
    # data in shape(no. of sims, no. of steps, 5), 5 for x, y, r, boltzmann factor, girsanov factor
    data = np.zeros((n_iters,5))
    
    # starting metad rerun simulation at the top-left basin
    starting_position = np.array([[-0.57,1.43,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = MullerForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Converged metadynamics bias
    bias_grid = Continuous1DFunction(bias[:,1],bias[:,0][0],bias[:,0][-1])
    cv = TorchForce(torchforce_savedir)
    biasforce = CustomCVForce('(1-1/{biasfactor})*bias_grid(cv)'.format(biasfactor=biasfactor))
    biasforce.addCollectiveVariable('cv',cv)
    biasforce.addTabulatedFunction('bias_grid',bias_grid)
    biasforce.setForceGroup(1)
    system.addForce(biasforce)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,:2] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0,:2]
        # Record Boltzmann weight
        bias_potential = simulation.context.getState(
            getEnergy=True,groups=0b00000000000000000000000000000010
        ).getPotentialEnergy().value_in_unit(kilojoules_per_mole)
        data[k,3] = np.exp(bias_potential / kt)
        # Step simulation
        simulation.step(nstxout)
        # Record Girsanov weight
        data[k,4] = simulation.integrator.getGlobalVariableByName("M")
    
    data[:,2] = fmrc.transform(tica.transform(data[:,:2])).reshape(-1,)
    
    np.save(save_dir,data)

#### 3. Metadynamics build-up simulation

##### 3.1 2d Muller Potential

In [ ]:
# Metadynamics parameters
biasfactor = 2
cv_sigma = 0.1
periodic_cv = False
gridwidth = 100
height = 1.2 * kilojoule_per_mole
pace = 20

In [ ]:
#%%time
# Number of repeats for error bars
n_repeats = 10
# Simulation length
n_iters = 100000

torchforce_savedir = 'models/muller_2d_torchforce.pt'
# Load the grid points so that this simulation is consistent with the rerun one
bias = np.load('bias/metad_rerun_bias.npy')

for n_re in range(n_repeats):
    print('Running {n_re}/{n_repeats} Repeated Experiment ...'.format(
        n_re = n_re, n_repeats = n_repeats
    ))

    save_dir = 'traj_and_dat/2d_muller/metad_build_up/{n_iters}_gamma={biasfactor}_metad_build_up_{n_re}.npy'.format(
        biasfactor = biasfactor, n_iters = n_iters, n_re = n_re
    )
    
    # data in shape(no. of sims, no. of steps, 5), 5 for x, y, r, boltzmann factor, girsanov factor
    data = np.zeros((n_iters,5))
    
    # starting metad rerun simulation at the top-left basin
    starting_position = np.array([[-0.57,1.43,0]])
    ### Re-build systems, since LangevinSplittingGirsanov has weird behaviour when
    ### using context.reinitialize()
    system = System()
    systemforce = MullerForce()
    system.addParticle(mass)
    systemforce.addParticle(0, [])
    system.addForce(systemforce)

    # Define the collective variable as x
    cv_force = TorchForce(torchforce_savedir)
    
    # Create the BiasVariable and Metadynamics objects
    cv = BiasVariable(
        cv_force, bias[:,0][0], bias[:,0][-1], cv_sigma, periodic_cv, gridwidth
    )
        
    metad = Metadynamics(
        system, [cv], temperature, biasfactor, height, pace#, save_freq, save_dir
    )
    
    # Set ForceGroup to 1 for Girsanov path weights
    metad._force.setForceGroup(1)

    topology = app.Topology()
    
    # Langevin integrator that allows girsanov reweighting factor output
    integrator = LangevinSplittingGirsanov(
        nstxout = nstxout, temperature = temperature,
        collision_rate = collision_rate, timestep = timestep,
        splitting = splitting
    )

    # Create simulation
    simulation = Simulation(topology,system,integrator,platform)
    simulation.context.setPositions(starting_position)
    simulation.context.setVelocitiesToTemperature(temperature)
    
    for k in range(n_iters):
        if k % 100 == 0:
            print(str(k)+'/'+str(n_iters))
        # Record Positions
        data[k,:2] = simulation.context.getState(
            getPositions=True
        ).getPositions(asNumpy=True).value_in_unit(nanometer)[0,:2]
        # Step simulation
        metad.step(simulation,nstxout)
        # Record Girsanov weight
        data[k,4] = simulation.integrator.getGlobalVariableByName("M")

    data[:,2] = fmrc.transform(tica.transform(data[:,:2])).reshape(-1,)
    # End-point bias potential reweighting: assuming pseudo-static bias
    # Retrieve the bias grid
    bias_grid = np.array(metad._table.getFunctionParameters()[0])
    cv_grid = np.linspace(bias[:,0][0],bias[:,0][-1],bias_grid.shape[0])
    # Re-construct the cubic spline
    spline = CubicSpline(cv_grid,bias_grid,bc_type='natural')
    data[:,3] = np.exp(spline(data[:,2])/kt)
    # For values outside the grid, the bias value should be 0
    data[:,3][np.where((data[:,2]<bias[:,0][0]) | (data[:,2] > bias[:,0][-1]))] = 1.0 
    np.save(save_dir,data)